<div style="font-family: 'Times New Roman'; font-weight: bold; font-size: 2em; text-align: center;">
    Analisis Pengaruh Teknik Preprocessing terhadap Klasifikasi Ginjal Normal dan Batu Ginjal Menggunakan Fitur Gray Level Co-occurrence Matrix (GLCM) serta Perbandingan Algoritma Support Vector Machine (SVM), Random Forest, dan K-Nearest Neighbor (KNN)
</div>

# Baseline Tanpa Preprocessing

In [1]:
import os
import csv
import math as _math
import cv2
import numpy as np
from tqdm import tqdm

## Import Library

Dilakukan import beberapa library yang diperlukan untuk mendukung proses ekstraksi fitur tekstur menggunakan metode GLCM. Library `os` digunakan untuk mengakses, membaca, dan mengelola struktur folder dataset citra sehingga program dapat melakukan pemrosesan gambar secara otomatis pada setiap kategori data. Library `csv` digunakan untuk menyimpan hasil ekstraksi fitur ke dalam file berformat CSV yang nantinya akan digunakan sebagai dataset masukan pada tahap klasifikasi. Library `math` yang diimpor dengan alias `_math` digunakan untuk melakukan perhitungan matematis, khususnya fungsi logaritma pada perhitungan fitur Entropy. Selanjutnya, library `cv2` (OpenCV) digunakan untuk membaca citra digital dan mengubahnya menjadi citra grayscale karena metode GLCM bekerja pada citra satu kanal intensitas. Library `numpy` digunakan untuk membuat dan memanipulasi matriks GLCM berukuran 256×256 serta melakukan berbagai operasi numerik yang dibutuhkan selama proses ekstraksi fitur. Terakhir, library `tqdm` digunakan untuk menampilkan progress bar selama proses ekstraksi berlangsung sehingga memudahkan pemantauan perkembangan pemrosesan seluruh citra dalam dataset.

In [2]:
PATH_INPUT = "Assets/"

KATEGORI = [
    "Normal",
    "kidneyStone"
]

OUTPUT_CSV = "hasil_ekstraksi_baseline.csv"

GLCM_DISTANCE = 1
GLCM_ANGLES = [0, 45, 90, 135]

## Konfigurasi Program

Dilakukan konfigurasi parameter utama yang akan digunakan selama proses ekstraksi fitur GLCM. Variabel `PATH_INPUT` digunakan untuk menentukan lokasi folder dataset yang berisi citra ginjal normal dan citra batu ginjal. Variabel `KATEGORI` berisi daftar kelas yang digunakan dalam project, yaitu `Normal` dan `kidneyStone`, sehingga program dapat membaca dan memproses citra berdasarkan kategori masing-masing. Variabel `OUTPUT_CSV` digunakan untuk menentukan nama file keluaran yang akan menyimpan seluruh hasil ekstraksi fitur tekstur dalam format CSV. Selanjutnya, parameter `GLCM_DISTANCE` ditetapkan bernilai 1 yang menunjukkan jarak antar piksel yang digunakan dalam pembentukan pasangan piksel pada matriks GLCM. Nilai ini dipilih karena merupakan jarak yang paling umum digunakan untuk menangkap hubungan tekstur lokal pada citra. Variabel `GLCM_ANGLES` berisi empat orientasi sudut, yaitu 0°, 45°, 90°, dan 135°, yang digunakan untuk merepresentasikan hubungan spasial piksel dari berbagai arah.

In [3]:
def build_glcm_manual(img, distance, angle_deg, levels=256):
    h, w = img.shape
    glcm = np.zeros((levels, levels), dtype=np.float64)

    if angle_deg == 0:
        di, dj = 0, distance
    elif angle_deg == 45:
        di, dj = -distance, distance
    elif angle_deg == 90:
        di, dj = -distance, 0
    elif angle_deg == 135:
        di, dj = -distance, -distance

    for i in range(h):
        for j in range(w):
            ni = i + di
            nj = j + dj
            if 0 <= ni < h and 0 <= nj < w:
                glcm[int(img[i, j]), int(img[ni, nj])] += 1

    glcm = glcm + glcm.T
    total = glcm.sum()
    if total > 0:
        glcm /= total
    return glcm

## Pembentukan Matriks GLCM

Dilakukan pembentukan matriks GLCM secara manual tanpa menggunakan fungsi bawaan seperti `graycomatrix()` dari pustaka scikit-image. Fungsi `build_glcm_manual()` menerima citra grayscale, jarak antar piksel (`distance`), sudut orientasi (`angle_deg`), serta jumlah tingkat keabuan (`levels`) sebagai masukan. Pertama, program mengambil ukuran citra dan membuat matriks GLCM berukuran 256×256 yang akan digunakan untuk menyimpan frekuensi kemunculan pasangan nilai intensitas piksel. Selanjutnya, ditentukan arah pergeseran piksel berdasarkan sudut yang digunakan, yaitu 0°, 45°, 90°, dan 135°. Setelah arah ditentukan, program melakukan penelusuran terhadap seluruh piksel pada citra dan mencari pasangan piksel tetangganya sesuai jarak dan sudut yang telah ditentukan. Setiap pasangan nilai intensitas yang ditemukan akan dicatat ke dalam matriks GLCM dengan menambahkan nilai frekuensinya pada posisi yang sesuai. Setelah seluruh pasangan piksel diproses, matriks dibuat simetris dengan menjumlahkan matriks dengan transposenya agar hubungan antar piksel pada kedua arah dapat terwakili. Tahap terakhir adalah normalisasi matriks dengan membagi setiap elemen terhadap jumlah seluruh frekuensi sehingga total nilai matriks menjadi 1. Hasil normalisasi ini menghasilkan matriks probabilitas yang kemudian digunakan sebagai dasar perhitungan berbagai fitur tekstur seperti Contrast, Homogeneity, Dissimilarity, Entropy, ASM, Energy, dan Correlation pada tahap berikutnya.

In [4]:
def ekstrak_fitur_glcm(glcm, levels=256):
    contrast = homogeneity = dissimilarity = entropy = asm = correlation = 0.0

    # Mean untuk korelasi
    mu_i = mu_j = 0.0
    for i in range(levels):
        for j in range(levels):
            mu_i += i * glcm[i, j]
            mu_j += j * glcm[i, j]

    # Standar deviasi untuk korelasi
    sigma_i = sigma_j = 0.0
    for i in range(levels):
        for j in range(levels):
            sigma_i += (i - mu_i) ** 2 * glcm[i, j]
            sigma_j += (j - mu_j) ** 2 * glcm[i, j]
    sigma_i = sigma_i ** 0.5
    sigma_j = sigma_j ** 0.5

    for i in range(levels):
        for j in range(levels):
            p = glcm[i, j]
            if p == 0:
                continue
            diff = i - j
            contrast      += (diff ** 2) * p
            homogeneity   += p / (1.0 + diff ** 2)
            dissimilarity += abs(diff) * p
            entropy       -= p * _math.log(p, 2)
            asm           += p ** 2
            if sigma_i > 0 and sigma_j > 0:
                correlation += ((i - mu_i) * (j - mu_j) * p) / (sigma_i * sigma_j)

    energy = asm ** 0.5
    return contrast, homogeneity, dissimilarity, entropy, asm, energy, correlation


## Ekstraksi Fitur Tekstur GLCM

Dilakukan perhitungan fitur-fitur tekstur berdasarkan matriks GLCM yang telah dibentuk pada tahap sebelumnya. Fungsi `ekstrak_fitur_glcm()` bertujuan untuk mengekstraksi tujuh fitur tekstur utama, yaitu Contrast, Homogeneity, Dissimilarity, Entropy, Angular Second Moment (ASM), Energy, dan Correlation. Sebelum menghitung Correlation, terlebih dahulu dihitung nilai rata-rata (mean) untuk baris dan kolom matriks GLCM. Nilai mean ini digunakan untuk mengetahui pusat distribusi probabilitas pasangan piksel pada matriks. Selanjutnya dihitung standar deviasi yang menggambarkan tingkat penyebaran nilai terhadap rata-ratanya. Setelah mean dan standar deviasi diperoleh, program melakukan iterasi terhadap seluruh elemen matriks GLCM yang telah dinormalisasi untuk menghitung masing-masing fitur tekstur. Fitur Contrast digunakan untuk mengukur tingkat perbedaan intensitas antar piksel bertetangga, Homogeneity mengukur tingkat keseragaman tekstur, Dissimilarity mengukur ketidaksamaan antar pasangan piksel, Entropy mengukur tingkat ketidakteraturan atau kompleksitas tekstur, ASM mengukur tingkat keseragaman distribusi probabilitas, Energy merupakan akar kuadrat dari ASM yang menunjukkan kekuatan tekstur, sedangkan Correlation mengukur hubungan linear antara pasangan piksel yang berdekatan.

In [5]:
def proses_glcm_satu_gambar(img, distance=1, angles=[0, 45, 90, 135]):
    row = {}
    for angle in angles:
        glcm = build_glcm_manual(img, distance, angle)
        c, h, d, e, a, en, cor = ekstrak_fitur_glcm(glcm)
        row[f"Contrast{angle}"]      = c
        row[f"Homogeneity{angle}"]   = h
        row[f"Dissimilarity{angle}"] = d
        row[f"Entropy{angle}"]       = e
        row[f"ASM{angle}"]           = a
        row[f"Energy{angle}"]        = en
        row[f"Correlation{angle}"]   = cor
    return row

## Proses Ekstraksi Fitur GLCM pada Satu Citra

Dibuat fungsi `proses_glcm_satu_gambar()` yang bertugas mengintegrasikan proses pembentukan matriks GLCM dan ekstraksi fitur tekstur untuk satu citra secara lengkap. Fungsi menerima citra grayscale sebagai masukan, kemudian membentuk matriks GLCM berdasarkan jarak piksel (`distance`) dan empat orientasi sudut yang digunakan dalam percobaan, yaitu 0°, 45°, 90°, dan 135°. Untuk setiap sudut, fungsi memanggil `build_glcm_manual()` guna membentuk matriks GLCM yang telah dinormalisasi, kemudian matriks tersebut diproses menggunakan fungsi `ekstrak_fitur_glcm()` untuk memperoleh tujuh fitur tekstur berupa Contrast, Homogeneity, Dissimilarity, Entropy, ASM, Energy, dan Correlation. Hasil fitur dari masing-masing sudut disimpan ke dalam struktur data dictionary dengan penamaan yang mencantumkan jenis fitur dan sudut orientasinya, misalnya `Contrast0`, `Contrast45`, `Contrast90`, dan `Contrast135`. Dengan menggunakan empat orientasi sudut, informasi tekstur dapat direpresentasikan dari berbagai arah sehingga karakteristik pola tekstur citra ginjal dapat ditangkap secara lebih komprehensif. Pada akhirnya, fungsi mengembalikan satu dictionary yang berisi seluruh fitur tekstur hasil ekstraksi dari satu citra. Karena terdapat tujuh fitur dan empat sudut orientasi, maka setiap citra menghasilkan total 28 fitur yang selanjutnya digunakan sebagai atribut masukan pada proses klasifikasi citra ginjal normal dan batu ginjal.

In [6]:
CSV_HEADER = ["Filename", "Label"]

for _angle in GLCM_ANGLES:
    for _fitur in [
        "Contrast",
        "Homogeneity",
        "Dissimilarity",
        "Entropy",
        "ASM",
        "Energy",
        "Correlation"
    ]:
        CSV_HEADER.append(f"{_fitur}{_angle}")

## Pembuatan Header Dataset Fitur

Dilakukan pembuatan struktur header yang akan digunakan sebagai nama kolom pada file CSV hasil ekstraksi fitur. Variabel `CSV_HEADER` diawali dengan dua kolom utama, yaitu `Filename` untuk menyimpan nama file citra dan `Label` untuk menyimpan kategori kelas citra, yaitu Normal atau Kidney Stone. Selanjutnya program melakukan perulangan terhadap setiap sudut orientasi GLCM yang digunakan, yaitu 0°, 45°, 90°, dan 135°. Pada setiap sudut tersebut, program kembali melakukan perulangan untuk menambahkan nama-nama fitur tekstur yang akan diekstraksi, yaitu Contrast, Homogeneity, Dissimilarity, Entropy, ASM, Energy, dan Correlation. Nama fitur kemudian digabungkan dengan nilai sudut sehingga menghasilkan nama kolom yang unik, seperti `Contrast0`, `Contrast45`, `Contrast90`, `Contrast135`, serta kolom-kolom fitur lainnya untuk setiap orientasi sudut. Proses ini dilakukan secara otomatis agar struktur dataset yang dihasilkan konsisten dengan jumlah fitur yang diekstraksi dari setiap citra. Karena terdapat tujuh fitur tekstur yang dihitung pada empat sudut orientasi berbeda, maka total kolom fitur yang terbentuk adalah 28 kolom, ditambah dua kolom identitas data yaitu `Filename` dan `Label`.

## Proses Ekstraksi Fitur Seluruh Dataset

Dilakukan proses ekstraksi fitur GLCM untuk seluruh citra yang terdapat pada dataset. Program diawali dengan menampilkan informasi bahwa proses ekstraksi fitur sedang berjalan dan membuat list `semua_baris` sebagai wadah untuk menyimpan seluruh hasil ekstraksi fitur dari setiap citra. Selanjutnya program melakukan perulangan terhadap setiap kategori yang telah ditentukan sebelumnya, yaitu kelas `Normal` dan `kidneyStone`. Untuk setiap kategori, program akan mengakses folder dataset yang sesuai dan melakukan pengecekan keberadaan folder tersebut. Jika folder tidak ditemukan, program akan menampilkan pesan peringatan dan melanjutkan proses ke kategori berikutnya.

Setelah folder ditemukan, program membaca seluruh file citra yang terdapat di dalam folder tersebut dan mengurutkannya menggunakan fungsi `sorted()` agar urutan pemrosesan data menjadi konsisten. Hanya file dengan ekstensi `.jpg`, `.jpeg`, dan `.png` yang akan diproses. Setiap citra kemudian dibaca dalam format grayscale menggunakan OpenCV karena metode GLCM memerlukan citra satu kanal intensitas. Apabila terjadi kegagalan saat membaca citra, data tersebut akan dilewati dan program melanjutkan proses ke citra berikutnya.

Selanjutnya fungsi `proses_glcm_satu_gambar()` dipanggil untuk membentuk matriks GLCM dan menghitung seluruh fitur tekstur pada empat orientasi sudut yang telah ditentukan. Hasil ekstraksi fitur yang diperoleh kemudian disimpan ke dalam sebuah dictionary yang berisi nama file citra (`Filename`), label kelas (`Label`), serta seluruh fitur tekstur hasil ekstraksi. Dictionary tersebut kemudian ditambahkan ke dalam list `semua_baris` sehingga seluruh data fitur dari semua citra terkumpul dalam satu struktur data. Setelah seluruh citra selesai diproses, program menampilkan jumlah total data yang berhasil diekstraksi.

In [7]:
print("\nMemulai ekstraksi fitur GLCM ...")
semua_baris = []

for label in KATEGORI:
    folder = os.path.join(PATH_INPUT, label)
    if not os.path.exists(folder):
        print(f"Folder tidak ditemukan: {folder}")
        continue
    for nama_file in tqdm(sorted(os.listdir(folder)), desc=label):
        if not nama_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        jalur = os.path.join(folder, nama_file)
        img = cv2.imread(jalur, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        fitur_row = proses_glcm_satu_gambar(img, GLCM_DISTANCE, GLCM_ANGLES)
        baris = {"Filename": nama_file, "Label": label}
        baris.update(fitur_row)
        semua_baris.append(baris)
        
print("Jumlah data:", len(semua_baris))


Memulai ekstraksi fitur GLCM ...


kidneyStone: 100%|██████████| 100/100 [01:43<00:00,  1.04s/it]

Jumlah data: 200


## Penyimpanan Hasil Ekstraksi Fitur ke File CSV

Dilakukan penyimpanan seluruh hasil ekstraksi fitur tekstur ke dalam file berformat CSV. Program menggunakan perintah `with open()` untuk membuat atau membuka file keluaran sesuai nama yang telah ditentukan pada variabel `OUTPUT_CSV`. Parameter `"w"` digunakan untuk menulis data baru ke dalam file, sedangkan `newline=""` digunakan agar tidak muncul baris kosong tambahan ketika file CSV dibuka pada aplikasi spreadsheet seperti Microsoft Excel. Selanjutnya objek `DictWriter` digunakan untuk menuliskan data yang berbentuk dictionary ke dalam file CSV dengan struktur kolom yang telah didefinisikan sebelumnya pada variabel `CSV_HEADER`. Program kemudian menuliskan baris header menggunakan `writeheader()` dan menuliskan seluruh data hasil ekstraksi fitur yang tersimpan dalam list `semua_baris` menggunakan `writerows()`. Setelah proses penyimpanan selesai, program menampilkan informasi jumlah data yang berhasil disimpan beserta nama file tujuan.

In [8]:
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_HEADER)
    writer.writeheader()
    writer.writerows(semua_baris)

print(f"{len(semua_baris)} data tersimpan ke: {OUTPUT_CSV}")

print(f"\n Selesai! CSV tersimpan di: {OUTPUT_CSV}")

200 data tersimpan ke: hasil_ekstraksi_baseline.csv

 Selesai! CSV tersimpan di: hasil_ekstraksi_baseline.csv
